# Lab 2 — Spark execution: partitions, shuffles and the UI

Five exercises on how Spark actually runs a job: how work is split into partitions, which
operations force data to move across the cluster, and how to read the evidence for both in
the Spark UI.

**Setup:** `getting_started/local_setup_mac.md` (or `local_setup_windows.md`), then from the
repo root:

```sh
uv sync
uv run python get_data.py --dataset taxi
uv run jupyter lab          # then open labs/L2/lab.ipynb
```

VS Code works too — open this notebook and pick `.venv/bin/python` as the kernel.

Run the cells in order. Exercises 3 and 5 send you to the Spark UI, which only exists while
the session is alive, so leave the final `spark.stop()` cell until you have captured both
screenshots.

The code is mostly given; the **written answers** are what is marked.

In [ ]:
import contextlib
import io
import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root() -> Path:
    """The directory holding get_data.py, searching upward from the working directory.

    A kernel starts in the notebook's own directory (labs/L2/), so walk up rather than
    assuming anything about where you launched Jupyter from.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "get_data.py").exists():
            return candidate
    raise RuntimeError("Could not find the repo root — open this notebook from inside the repo.")


# Everything below assumes the repo root as the working directory: the dataset path, and the
# spark-warehouse/ and metastore_db/ directories Spark creates next to it. Move there once,
# here, before Spark starts.
os.chdir(find_repo_root())
print(f"working directory: {Path.cwd()}")

# A kernel started from an IDE never sources your shell profile, so JAVA_HOME can be missing
# here even though `java -version` works fine in a terminal. Resolve it before PySpark loads.
if "JAVA_HOME" not in os.environ:
    if sys.platform == "darwin":
        os.environ["JAVA_HOME"] = subprocess.check_output(["brew", "--prefix", "openjdk@17"], text=True).strip()
    else:
        raise RuntimeError("Set JAVA_HOME for your user account, then restart the kernel.")
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

# Imported after the block above on purpose: PySpark looks for the JVM at import time.
from pyspark.sql import SparkSession  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

In [ ]:
# local[4] rather than local[*]: this lab measures parallelism, so the number of slots has
# to be a number you know rather than "however many cores this laptop has".
CORES = 4

spark = (
    SparkSession.builder.appName("L2-execution")
    .master(f"local[{CORES}]")
    # A persistent catalog (a local Derby metastore under metastore_db/). Without it Spark
    # forgets its tables when the kernel dies but leaves their files in spark-warehouse/,
    # and the *second* run of this notebook fails with LOCATION_ALREADY_EXISTS.
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version}, master {spark.sparkContext.master}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


def physical_plan(df) -> str:
    """The physical plan as text.

    `df.explain()` prints to stdout rather than returning a string, so capture it. Using the
    public API keeps this lab to vanilla PySpark — no reaching into `df._jdf`.
    """
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

## Setup — build the working table

One month of NYC taxi trips, read from Parquet and saved as a local table.

In [ ]:
# Lets turn off adaptive query execution (AQE) for now (we will explore it efffect on performance a bit later).
spark.conf.set("spark.sql.adaptive.enabled", False)

In [ ]:
# Relative to the repo root, which the setup cell made the working directory.
SRC = "data/yellow_tripdata_2023-01.parquet"
if not Path(SRC).exists():
    raise SystemExit(f"Missing {SRC}\nRun: uv run python get_data.py --dataset taxi")

# Name the work so it is identifiable in the Spark UI. The label sticks to every job this
# kernel starts until it is changed, so each cell below sets its own.
spark.sparkContext.setJobDescription("Setup: read trips data")

raw = spark.read.parquet(SRC)

trips = raw.select(
    F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"),
    F.col("passenger_count").cast("int").alias("passenger_count"),
    F.col("trip_distance").cast("double").alias("trip_distance"),
    F.col("fare_amount").cast("double").alias("fare_amount"),
    F.col("tip_amount").cast("double").alias("tip_amount"),
    F.col("payment_type").cast("int").alias("payment_type"),
    F.col("PULocationID").cast("int").alias("pu_location_id"),
)

Navigate to Spark UI, we should see now one job. And you may say, well, but spark is lazy, how that is possible? 
The answer is because spark.read.parquet() has to return a DataFrame that already knows its columns — and finding them out requires reading the file. Laziness applies to data, not to metadata.

Next, lest examine the 'effect' of how data is saved. 
First, we let spark to figure out how to save it, then we try forcing to split the data into 8 pieces and save it to disk.

In [ ]:
spark.sparkContext.setJobDescription("Setup: prepare trips default table")
spark.sql("DROP TABLE IF EXISTS trips")

spark.sparkContext.setJobDescription("Setup: save trips data as table")
trips.write.mode("overwrite").saveAsTable("trips")

In [ ]:
spark.sparkContext.setJobDescription("Setup: prepare trips_repartitioned table")
spark.sql("DROP TABLE IF EXISTS trips_repartitioned")

spark.sparkContext.setJobDescription("Setup: save trips data as table (repartitioned to 8 partitions)")
trips.repartition(8).write.mode("overwrite").saveAsTable("trips_repartitioned")

In [ ]:
from utils import get_tbl_partitions_summary


spark.sparkContext.setJobDescription("Inspect: rows per file in trips table")
_ = get_tbl_partitions_summary("trips")

In [ ]:
spark.sparkContext.setJobDescription("Inspect: rows per file in trips_repartitioned table")
_ = get_tbl_partitions_summary("trips_repartitioned")

! The two reports don't agree, and that's the point. Files and partitions are set by two different mechanisms that never consult each other. The write chose 8 (you asked for repartition(8)). The read recomputes from scratch: it ignores how many files there are and packs them into splits sized for the cluster. Here that packing puts exactly two files in each split.

## Exercise 1 — Partitions and parallelism

A partition is the unit of work. A slot processes one partition at a time.

Too few partitions: slots sit idle. Too many: you pay scheduling overhead per task and
produce a pile of tiny output files.

**Before you run anything**, write down your prediction: as partition count goes
1 → 8 → 64 → 512 → 4096, what shape does the runtime curve have?

**Your prediction:**

_(replace this line)_

In [ ]:
def timed_agg(n_partitions: int) -> float:
    """Repartition, run a fixed aggregation, return wall-clock seconds."""
    # Name each run after its partition count. Without this the five runs are identical in
    # the Spark UI and you cannot tell which job produced which timing.
    spark.sparkContext.setJobDescription(f"Ex1: repartition({n_partitions}) then aggregate")
    t0 = time.time()
    (
        trips.repartition(n_partitions)
        .groupBy("payment_type")
        .agg(F.avg("fare_amount"))
        .write.format("noop")
        .mode("overwrite")
        .save()  # forces execution, writes nothing
    )
    return time.time() - t0


# Drop the last one or two values if your laptop is slow — the shape is what matters.
SWEEP = [1, 8, 64, 512, 4096]

results = []
for n in SWEEP:
    secs = timed_agg(n)
    results.append((n, round(secs, 2)))
    print(f"{n:>5} partitions -> {secs:6.2f} s")

In [ ]:
spark.sparkContext.setJobDescription("Ex1: summary of the partition sweep")
spark.createDataFrame(results, "partitions int, seconds double").show()

**Q1a.** Was your prediction right? Where is the minimum, and why is the curve higher on
*both* sides of it?

_(replace this line)_

**Q1b.** `spark.sql.shuffle.partitions` defaults to 200. Print it below. Given your
measurements, is 200 sensible for *this* dataset on *this* machine? What would you set it
to, and what information did you need that the default could not have?

_(replace this line)_

In [ ]:
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("cores available to this session:", CORES)

## Exercise 2 — Narrow vs wide

Classify each of the six operations below as **narrow** (no data moves between executors)
or **wide** (a shuffle). Fill in your answer *first*, then run the verification cell.

In [ ]:
# TODO: replace each None with "narrow" or "wide"
answers = {
    "filter": None,
    "withColumn": None,
    "groupBy+agg": None,
    "orderBy": None,
    "select": None,
    "dropDuplicates": None,
}
answers

In [ ]:
ops = {
    "filter": trips.filter(F.col("fare_amount") > 10),
    "withColumn": trips.withColumn("tip_pct", F.col("tip_amount") / F.col("fare_amount")),
    "groupBy+agg": trips.groupBy("payment_type").count(),
    "orderBy": trips.orderBy("fare_amount"),
    "select": trips.select("fare_amount", "trip_distance"),
    "dropDuplicates": trips.dropDuplicates(["pu_location_id"]),
}

# An `Exchange` node in the physical plan IS a shuffle. That is the whole test.
print(f"{'operation':<16} {'you said':<10} {'plan says':<10}")
print("-" * 38)
for name, df in ops.items():
    truth = "wide" if "Exchange" in physical_plan(df) else "narrow"
    mark = "ok" if answers[name] == truth else "<-- check"
    print(f"{name:<16} {str(answers[name]):<10} {truth:<10} {mark}")

**Q2.** Which one surprised you, and what is the underlying reason it behaves that way?

_(replace this line)_

## Exercise 3 — Reading the Spark UI

Run the query below, then open the Spark UI. The URL was printed by the session cell —
usually <http://localhost:4040>.

**Finding your job.** By default the Jobs tab lists work by call site —
`count at NativeMethodAccessorImpl.java:0` and similar — which tells you nothing once a dozen
cells have run. Every cell in this notebook that starts work first calls

```python
spark.sparkContext.setJobDescription("Ex3: trips per hour and payment type")
```

and that string is what you see in the **Description** column of the Jobs tab, and in the
SQL/DataFrame tab. So look for `Ex3:` to find the query you just ran.

Two things that will otherwise confuse you:

- The description is **sticky** — it applies to every job the kernel starts until something
  changes it. That is why each cell sets its own, and why running cells out of order can
  leave a job wearing the previous cell's name.
- **One action is usually several jobs.** With AQE on, Spark submits each query stage as its
  own job, so the query below appears as a handful of job IDs that all share one description.


In [ ]:
spark.sparkContext.setJobDescription("Ex3: trips per hour and payment type")

result = (
    trips.filter(F.col("fare_amount").between(0, 500))
    .withColumn("hour", F.hour("pickup_ts"))
    .groupBy("hour", "payment_type")
    .agg(
        F.count("*").alias("trips"),
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("tip_amount").alias("avg_tip"),
    )
    .orderBy("hour", "payment_type")
)

result.show(10)

In the Spark UI, from the **Stages** tab:

**Q3a.** How many stages did this job have, and what separated them?

_(replace this line)_

**Q3b.** Which stage took the longest? Report its duration and its shuffle read/write size.

_(replace this line)_

**Q3c.** Look at the task duration summary for that stage (min / median / max). Are the
tasks balanced? What does the gap between median and max tell you?

_(replace this line)_

**Paste a screenshot of the Stages tab below.**

_(screenshot here)_

## Exercise 4 — Adaptive Query Execution

AQE re-plans a query *at runtime*, using statistics from completed stages instead of
estimates made before anything ran.

In [ ]:
def build_query():
    """Build the query fresh.

    This has to be a function. A DataFrame caches its QueryExecution the first time it is
    planned, so reusing one object across a `spark.conf.set` gives you the *old* plan back
    and the comparison below silently shows no difference.
    """
    return (
        trips.filter(F.col("payment_type") == 2)  # a small slice
        .groupBy("pu_location_id")
        .agg(F.avg("fare_amount").alias("avg_fare"))
    )

In [ ]:
plans = {}
for aqe in (False, True):
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    q = build_query()
    # Q4a asks you to compare the task counts of these two runs. Labelling them separately is
    # what makes them tellable apart in the UI.
    spark.sparkContext.setJobDescription(f"Ex4: avg fare by location (AQE {'on' if aqe else 'off'})")
    t0 = time.time()
    # collect() rather than a noop write: it runs *this* DataFrame's query execution, so the
    # plan we read back afterwards is the final adaptive one (`isFinalPlan=true`) rather than
    # the initial guess. The result is one row per pickup location — a couple of hundred.
    rows = q.collect()
    elapsed = time.time() - t0
    plans[aqe] = physical_plan(q)
    print(f"AQE {'on ' if aqe else 'off'}: {elapsed:5.2f} s, {len(rows)} rows")

In [ ]:
for aqe, plan in plans.items():
    print(
        f"AQE {'on ' if aqe else 'off'}  "
        f"AdaptiveSparkPlan={'AdaptiveSparkPlan' in plan!s:<5}  "
        f"AQEShuffleRead={'AQEShuffleRead' in plan}"
    )

print()
print(plans[True][:1200])

**Q4a.** With AQE off the plan is a plain physical plan; with AQE on it is wrapped in an
`AdaptiveSparkPlan` node marked `isFinalPlan=true`. Confirm that above. Then look at the
task counts for each run in the Spark UI — how many shuffle partitions did each actually
use, and how does that compare to the `spark.sql.shuffle.partitions` you printed earlier?

_(replace this line)_

**Q4b.** An `AQEShuffleRead` node only appears once AQE has actually *coalesced* something.
Find it in the plan text above and say what it did: how many partitions went in, how many
came out, and why AQE was able to make that call at runtime when the initial planner
could not.

_(replace this line)_

**Q4c.** AQE has three main tricks: coalescing shuffle partitions, switching join
strategies, and splitting skewed partitions. Which one fired here, and how do you know?

_(replace this line)_

## Exercise 5 — Skew

Skew is when one partition holds far more data than the others. The stage cannot finish
until its slowest task does, so one fat partition stalls everything.

We build a deliberately skewed key and join on it.

In [ ]:
# 90% of rows get key 0; the rest are spread over 1..99
skewed = trips.withColumn(
    "join_key",
    F.when(F.rand(seed=42) < 0.9, F.lit(0)).otherwise((F.rand(seed=7) * 99 + 1).cast("int")),
)

lookup = (
    spark.range(0, 100)
    .withColumnRenamed("id", "join_key")
    .withColumn("label", F.concat(F.lit("key_"), F.col("join_key")))
)

spark.sparkContext.setJobDescription("Ex5: row count per join key (confirm the skew)")
skewed.groupBy("join_key").count().orderBy(F.desc("count")).show(5)

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # force a shuffle join

spark.sparkContext.setJobDescription("Ex5: skewed shuffle join (before mitigation)")

t0 = time.time()
(
    skewed.join(lookup, "join_key")
    .groupBy("label")
    .agg(F.avg("fare_amount"))
    .write.format("noop")
    .mode("overwrite")
    .save()
)
print(f"skewed shuffle join: {time.time() - t0:.2f} s")

Open the Spark UI for that job and look at the task durations in the join stage.

**Q5a.** What is the ratio of max task duration to median task duration? What does the task
duration histogram look like?

_(replace this line)_

**Paste a screenshot of the task summary below.**

_(screenshot here)_

### Now fix it

Three options. Try at least one, measure it, and say why it worked.

1. **Broadcast the small side** — `lookup` has 100 rows. Broadcasting removes the shuffle
   entirely. Re-enable `autoBroadcastJoinThreshold`, or use `F.broadcast()`.
2. **Let AQE handle it** — turn adaptive execution back on and let it split the skewed
   partition.
3. **Salt the key** — add a random suffix to the hot key on the large side, replicate the
   small side across those salt values, join on the salted key.

In [ ]:
# TODO: apply a mitigation and time it
# The label is already set so this run sits next to the un-mitigated one in the Spark UI,
# ready for the side-by-side comparison Q5b asks for.
spark.sparkContext.setJobDescription("Ex5: mitigated join")

t0 = time.time()
# ... your code ...
print(f"mitigated: {time.time() - t0:.2f} s")

**Q5b.** Which mitigation did you use, what was the speedup, and *why* did it work? Be
specific about what stopped happening.

_(replace this line)_

**Q5c.** Broadcast is the obvious fix here because the lookup table is tiny. Describe a
situation where broadcast is not available and salting is the only option left.

_(replace this line)_

In [ ]:
# restore defaults
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.adaptive.enabled", True)

## Appendix — where does a table actually live?

*Not marked. Everything above was about how Spark **runs** a query; this is about where the
table it read from is **kept**, because the two are answered by completely different machinery.*

Every table in Spark has a three-part name:

```
catalog . database . table
```

- **catalog** — a pluggable source of metadata. You have been using `spark_catalog`, the
  built-in one, without naming it. It is the only catalog Spark ships enabled.
- **database** — a namespace inside a catalog, also called a *schema*. Yours is `default`.
- **table** — the thing itself.

So the `trips` table you built in the setup cell is really `spark_catalog.default.trips`.
Run the cell below to confirm it.

In [ ]:
spark.sparkContext.setJobDescription("Appendix: inspect the catalog")

print("catalogs :", [c.name for c in spark.catalog.listCatalogs()])
print("current  :", spark.catalog.currentCatalog(), "/", spark.catalog.currentDatabase())
print("databases:", [d.name for d in spark.catalog.listDatabases()])
print()

# The fully qualified name works exactly like the short one.
print("trips row count via the full name:", spark.table("spark_catalog.default.trips").count())
print()

# DESCRIBE EXTENDED is how you ask the catalog what it knows about a table.
for row in spark.sql("DESCRIBE EXTENDED trips").collect():
    if row[0] in ("Catalog", "Database", "Table", "Type", "Provider", "Location"):
        print(f"  {row[0]:<10} {row[1]}")

### Two directories, two different jobs

Note the `Location` printed above: it points inside `spark-warehouse/`. That is the split
worth understanding, because the two directories in your repo root are not variations on the
same idea.

| | `metastore_db/` | `spark-warehouse/` |
|---|---|---|
| Stores | *that* the table exists — its name, columns, format, location | the rows themselves |
| Format | an embedded Apache Derby database (Java's SQLite) | Parquet files |
| Controlled by | `javax.jdo.option.ConnectionURL` | `spark.sql.warehouse.dir` |

Both are defaults belonging to `spark_catalog`, not laws of Spark. The name `metastore_db`
comes from a string compiled into the Hive jar that ships inside PySpark —
`jdbc:derby:;databaseName=metastore_db;create=true` — which is why it appears in whatever
directory the driver happens to start in.

They are two halves of one thing, so **delete them together or not at all**. Remove
`metastore_db/` alone and Spark forgets the table while its files remain, and the next
`saveAsTable` fails with `LOCATION_ALREADY_EXISTS` — a directory occupied by a table nobody
remembers. Remove `spark-warehouse/` alone and you get the mirror image: a catalogue entry
pointing at nothing.

### Storage is per-database, not global

`spark.sql.warehouse.dir` is only the *default* location. A database can be given its own.

In [ ]:
spark.sparkContext.setJobDescription("Appendix: database with a custom location")

CUSTOM_DIR = str(Path.cwd() / "custom_warehouse")

spark.sql(f"CREATE DATABASE IF NOT EXISTS taxi_db LOCATION '{CUSTOM_DIR}'")

# A small summary of the trips data — a couple of hundred rows, not three million.
hourly = trips.groupBy(F.hour("pickup_ts").alias("hour"), "payment_type").agg(
    F.count("*").alias("trip_count"), F.avg("fare_amount").alias("avg_fare")
)
hourly.write.mode("overwrite").saveAsTable("taxi_db.hourly_summary")

location = [r[1] for r in spark.sql("DESCRIBE EXTENDED taxi_db.hourly_summary").collect() if r[0] == "Location"][0]
print("default database  ->", [r[1] for r in spark.sql("DESCRIBE EXTENDED trips").collect() if r[0] == "Location"][0])
print("taxi_db           ->", location)
print()
print("Same catalog, same metastore_db — only the bytes moved.")

### A second catalog

A database moves the *data*. A **catalog** replaces the metadata layer itself — where Spark
looks tables up in the first place. This is the Catalog Plugin API, and it is how Delta Lake,
Iceberg and Unity Catalog attach to Spark.

You can register one without rebuilding the session: `spark.sql.catalog.<name>` names the
implementation class, and `spark.sql.catalog.<name>.*` configures it. Below we use
`JDBCTableCatalog`, which keeps its metadata *and* its data in a JDBC database — pointed at
its own embedded Derby instance, entirely separate from `metastore_db`.

In [ ]:
spark.sparkContext.setJobDescription("Appendix: write into a custom catalog")

# Register a second catalog at runtime — no new SparkSession needed.
spark.conf.set("spark.sql.catalog.taxi_cat", "org.apache.spark.sql.execution.datasources.v2.jdbc.JDBCTableCatalog")
spark.conf.set("spark.sql.catalog.taxi_cat.url", "jdbc:derby:taxi_catalog_db;create=true")
spark.conf.set("spark.sql.catalog.taxi_cat.driver", "org.apache.derby.jdbc.EmbeddedDriver")

spark.sql("CREATE SCHEMA IF NOT EXISTS taxi_cat.curated")

# writeTo() is the DataFrameWriterV2 API. saveAsTable() resolves this name fine, but it
# issues CREATE TABLE ... USING parquet, and a JDBC catalog has no file provider to
# satisfy that — it rejects the command with NOT_SUPPORTED_IN_JDBC_CATALOG.
hourly.writeTo("taxi_cat.curated.hourly_summary").createOrReplace()

print("rows read back:", spark.table("taxi_cat.curated.hourly_summary").count())
print("catalogs now  :", [c.name for c in spark.catalog.listCatalogs()])
print("tables        :", [(r[0], r[1]) for r in spark.sql("SHOW TABLES IN taxi_cat.curated").collect()])
print()

# One query, both catalogs. Spark resolves each name against its own metadata source.
spark.sql("""
    SELECT w.hour, w.trip_count AS from_warehouse, c.trip_count AS from_custom_catalog
    FROM taxi_db.hourly_summary w
    JOIN taxi_cat.curated.hourly_summary c
      ON w.hour = c.hour AND w.payment_type = c.payment_type
    WHERE w.payment_type = 1
    ORDER BY w.hour
""").show(5)

In [ ]:
spark.sparkContext.setJobDescription("Appendix: what is on disk")

for name in ["metastore_db", "spark-warehouse", "custom_warehouse", "taxi_catalog_db"]:
    path = Path(name)
    if not path.exists():
        print(f"{name:<18} (absent)")
        continue
    parquet = len(list(path.rglob("*.parquet")))
    print(
        f"{name:<18} {sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / 1024:>8.0f} KB"
        f"   {parquet} parquet file(s)"
    )

### What that shows

Three storage locations, two metadata stores, one session:

| Table | Metadata lives in | Bytes live in |
|---|---|---|
| `spark_catalog.default.trips` | `metastore_db/` | `spark-warehouse/trips/` |
| `spark_catalog.taxi_db.hourly_summary` | `metastore_db/` | `custom_warehouse/` |
| `taxi_cat.curated.hourly_summary` | `taxi_catalog_db/` | `taxi_catalog_db/` |

The first two share a catalog and therefore a metastore, but not a directory — that is what
`CREATE DATABASE … LOCATION` did. The third shares neither: registering a catalog replaced
the whole lookup mechanism, so `metastore_db` and `spark-warehouse` were never consulted, and
the table is not stored as Parquet at all.

That is the point worth carrying to Databricks. `metastore_db` and `spark-warehouse` are not
"how Spark stores tables" — they are the defaults of one catalog implementation. Unity Catalog
is another implementation of the same plug-in point: a different metadata service, different
storage, the same `catalog.database.table` names in your SQL.

**Optional, not marked.** `spark_catalog.default.trips` and
`taxi_cat.curated.hourly_summary` are both reachable from one session by name alone. What
would you have to know, or do, to move a table from one to the other — and which of the two
directories would each step touch?

_(replace this line)_

To clean up everything this appendix created:

```sh
rm -rf custom_warehouse taxi_catalog_db
```

Both are git-ignored, as are `spark-warehouse/` and `metastore_db/`.

## Wrap-up

**Q6.** You have five exercises' worth of measurements from your own machine. If you had to
put this job into production tomorrow, what one thing would you change about how it runs —
and which single measurement from above drove that decision? Name the number.

_(replace this line)_

## Done

Submit `L2_<your-surname>.ipynb` with every *(replace this line)* filled in and both
screenshots in place.

In [ ]:
# This also shuts down the Spark UI, so finish exercises 3 and 5 before running it.
spark.stop()